# Once Upon a Video Store, in Python

In the game you helped a video store guess whether Vera, a newcomer, will like *Love Actually*. You
multiplied her ratings by each customer's, film by film, and added up the products: the **inner product**.
The customer with the largest one, Kai, is Vera's **nearest neighbour**, and the store copied Kai's rating.
Now you teach a computer to do the same, with three short Python functions (about 30 minutes):

1. `inner_product`: how much do two customers agree?
2. `nearest_customer`: which customer agrees most with a newcomer?
3. `predict`: copy that customer's rating of a film the newcomer has not seen.

**How it works.** You fill in the 🎯 cells; every other cell is ready to run. Run the cells from top to
bottom. A folded cell with a title holds plumbing: run it, and open it only if you are curious. After each
task, a check cell tells you whether your function works.

In [ ]:
#@title Setup (run this first) { display-mode: "form" }
# The plumbing of part 1: the table of the store, and the checks of tasks 1 to 3.
# You do not need to read it. A check prints one short message and never stops the notebook.
try:
    from IPython.display import display as _part1_display
except ImportError:
    _part1_display = None

_PART1_MARKS = {2: '✓✓', 1: '✓', 0: '~', -1: '✗', -2: '✗✗'}
_PART1_COLOURS = {2: "#3B82D6", 1: "#3B82D6", 0: "#8C8C8C", -1: "#E0742A", -2: "#E0742A"}
_PART1_LEGEND = "✓✓ 2 loved it, ✓ 1 liked it, ~ 0 neutral, ✗ -1 disliked it, ✗✗ -2 hated it, ? not seen yet"


def _part1_mark(rating, as_html):
    """One cell of the table: the mark and the number, or ? when the rating is not known."""
    if rating is None:
        return "?"
    mark = _PART1_MARKS.get(rating, "")
    if as_html:
        colour = _PART1_COLOURS.get(rating, "inherit")
        return '<b style="color:%s">%s</b>&nbsp;%s' % (colour, mark, rating)
    return "%-2s %2s" % (mark, rating)


def _part1_table(customers, films, ratings, newcomer, newcomer_ratings, new_film, new_film_ratings):
    """Show the store as a table of marks and numbers, with the newcomer in the last row."""
    from html import escape
    try:
        names = list(customers) + [newcomer]
        heads = list(films) + [new_film]
        rows = []
        for i in range(len(customers)):
            rows.append(list(ratings[i]) + [new_film_ratings[i]])
        rows.append(list(newcomer_ratings) + [None])

        # Plain text, for screens that cannot show the table
        name_width = max(len(name) for name in names) + 3
        widths = [max(len(head), 6) + 3 for head in heads]
        lines = [" " * name_width + "".join(heads[j].ljust(widths[j]) for j in range(len(heads)))]
        for r in range(len(rows)):
            cells = "".join(_part1_mark(rows[r][j], False).ljust(widths[j]) for j in range(len(heads)))
            lines.append(names[r].ljust(name_width) + cells)
        text = "\n".join(line.rstrip() for line in lines) + "\n\n" + _PART1_LEGEND

        # The same table in HTML: the last column (the film the newcomer has not seen) and the
        # newcomer's row are set apart by a stronger line
        thin = "border-bottom:1px solid rgba(128,128,128,0.35);"
        strong = "2px solid rgba(128,128,128,0.7);"
        html = '<table style="border-collapse:collapse;font-size:15px;margin:4px 0 6px 0">'
        html += '<tr><th style="padding:5px 12px;%s"></th>' % thin
        for j in range(len(heads)):
            left = "border-left:" + strong if j == len(heads) - 1 else ""
            html += '<th style="padding:5px 12px;text-align:center;%s%s">%s</th>' % (thin, left, escape(str(heads[j])))
        html += "</tr>"
        for r in range(len(rows)):
            is_newcomer = r == len(rows) - 1
            top = "border-top:" + strong if is_newcomer else ""
            name = escape(str(names[r]))
            if is_newcomer:
                name = "<b>%s</b>" % name
            html += '<tr><td style="padding:5px 12px;text-align:left;%s%s">%s</td>' % (thin, top, name)
            for j in range(len(heads)):
                left = "border-left:" + strong if j == len(heads) - 1 else ""
                html += '<td style="padding:5px 12px;text-align:center;%s%s%s">%s</td>' % (
                    thin, top, left, _part1_mark(rows[r][j], True))
            html += "</tr>"
        html += "</table>"
        html += '<div style="font-size:13px;opacity:0.8">%s</div>' % escape(_PART1_LEGEND)

        if _part1_display is None:
            print(text)
        else:
            _part1_display({"text/plain": text, "text/html": html}, raw=True)
    except Exception as error:
        print("The table could not be drawn (%s). Run the cell with the data again, then this one." % error)


# Each test: the inputs, the expected answer, the call as you would type it, and why that is the answer.
_PART1_TESTS = {
    "inner_product": [
        (([2, 1, -1], [1, 2, -2]), 6,
         'inner_product([2, 1, -1], [1, 2, -2])',
         'Vera and Kai: 2*1 + 1*2 + (-1)*(-2) = 6'),
        (([2, 1, -1], [2, 1, 2]), 3,
         'inner_product([2, 1, -1], [2, 1, 2])',
         'Vera and Pia: 2*2 + 1*1 + (-1)*2 = 3'),
        (([1, 2, -2], [2, 1, -1]), 6,
         'inner_product([1, 2, -2], [2, 1, -1])',
         'Kai and Vera, the other way round: 1*2 + 2*1 + (-2)*(-1) = 6'),
        (([2, 1, -1], [-2, -1, 2]), -7,
         'inner_product([2, 1, -1], [-2, -1, 2])',
         'Vera and Rosa: 2*(-2) + 1*(-1) + (-1)*2 = -7'),
        (([1, 1, -1, -1], [-1, -1, 1, 1]), -4,
         'inner_product([1, 1, -1, -1], [-1, -1, 1, 1])',
         'Lea and Max from round 1, with four films: 1*(-1) + 1*(-1) + (-1)*1 + (-1)*1 = -4'),
        (([2], [-2]), -4,
         'inner_product([2], [-2])',
         'One film only: 2*(-2) = -4'),
    ],
    "nearest_customer": [
        (([[2, -2, 0], [2, 1, 2], [-1, -2, -1], [1, 2, -2], [0, 1, 0], [-2, -1, 2]], [2, 1, -1]), 3,
         'nearest_customer(ratings, vera)',
         'Ivo, Pia, Tom, Kai, Jana and Rosa score 2, 3, -3, 6, 1 and -7 with Vera: Kai, in row 3, is the nearest'),
        (([[-1, -2, -1], [0, 1, 0], [1, 2, -2]], [2, 1, -1]), 2,
         'nearest_customer([[-1, -2, -1], [0, 1, 0], [1, 2, -2]], [2, 1, -1])',
         'Tom, Jana and Kai score -3, 1 and 6 with Vera: Kai, in row 2, is the nearest'),
        (([[1, 1, -1], [-1, -1, 1], [1, 1, -1], [-1, -1, 1]], [-1, -1, 1]), 1,
         'nearest_customer([[1, 1, -1], [-1, -1, 1], [1, 1, -1], [-1, -1, 1]], [-1, -1, 1])',
         'Round 1: Lea, Max, Nina and Omar score -3, 3, -3 and 3 with Zoe. Max (row 1) and Omar (row 3) tie, and on a tie the first one wins'),
        (([[-2, -1, 2], [-1, -2, -1]], [2, 1, -1]), 1,
         'nearest_customer([[-2, -1, 2], [-1, -2, -1]], [2, 1, -1])',
         'Rosa and Tom score -7 and -3 with Vera: both scores are below zero, and the larger one still wins, Tom in row 1'),
        (([[0, 1, 0]], [2, 1, -1]), 0,
         'nearest_customer([[0, 1, 0]], [2, 1, -1])',
         'With a single customer, row 0 is the nearest'),
    ],
    "predict": [
        (([[2, -2, 0], [2, 1, 2], [-1, -2, -1], [1, 2, -2], [0, 1, 0], [-2, -1, 2]], [1, 2, -1, -2, 0, 2], [2, 1, -1]), -2,
         'predict(ratings, love_actually, vera)',
         "Kai, in row 3, is Vera's nearest neighbour, and Kai's rating of Love Actually is -2"),
        (([[1, 1, -1], [-1, -1, 1], [1, 1, -1], [-1, -1, 1]], [-1, 1, -1, 1], [-1, -1, 1]), 1,
         'predict([[1, 1, -1], [-1, -1, 1], [1, 1, -1], [-1, -1, 1]], [-1, 1, -1, 1], [-1, -1, 1])',
         "Round 1: Max, in row 1, is Zoe's nearest neighbour (a tie with Omar, and the first wins), and Max's rating of Love Actually is 1"),
        (([[2, -2], [2, 1], [-1, -2], [1, 2], [0, 1], [-2, -1]], [0, 2, -1, -2, 0, 2], [2, 1]), 2,
         'predict([[2, -2], [2, 1], [-1, -2], [1, 2], [0, 1], [-2, -1]], [0, 2, -1, -2, 0, 2], [2, 1])',
         "Predicting Notting Hill from Jaws and The Shining: Pia, in row 1, is Vera's nearest neighbour (5), and Pia's rating of Notting Hill is 2"),
        (([[-2, 0], [1, 2], [-2, -1], [2, -2], [1, 0], [-1, 2]], [2, 2, -1, 1, 0, -2], [1, -1]), 1,
         'predict([[-2, 0], [1, 2], [-2, -1], [2, -2], [1, 0], [-1, 2]], [2, 2, -1, 1, 0, -2], [1, -1])',
         "Predicting Jaws from The Shining and Notting Hill: Kai, in row 3, is Vera's nearest neighbour (4), and Kai's rating of Jaws is 1"),
    ],
}

_PART1_HINTS = {
    "inner_product": "in the loop, multiply customer_a[i] by customer_b[i], and add the product to total.",
    "nearest_customer": "score is the inner product of ratings[i] with new_customer, and row i becomes "
                        "the best so far when its score is larger than best_score.",
    "predict": "nearest_customer(ratings, new_customer) gives the row of the nearest customer, and "
               "new_film[row] is that customer's rating of the new film.",
}


def _part1_call(fn, inputs):
    """Run a participant's function on fresh copies of the inputs, with its prints silenced.
    Gives back (answer, error), where error is None when the call went through. Never raises."""
    import contextlib, copy, io
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            answer = fn(*copy.deepcopy(inputs))
        return answer, None
    except Exception as error:
        return None, "%s: %s" % (type(error).__name__, error)


def _part1_first_failure(name, fn):
    """The first test of name that fn fails, or None when fn passes them all."""
    for inputs, expected, call, why in _PART1_TESTS[name]:
        answer, error = _part1_call(fn, inputs)
        try:
            right = bool(error is None and answer == expected)
        except Exception:
            right = False
        if not right:
            return {"answer": answer, "error": error, "expected": expected, "call": call, "why": why}
    return None


def _part1_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    # A gap such as "x = ..." or "return ..." stays in the compiled function, so no source is needed;
    # a gap such as "if ...:" is only visible in the source, which Jupyter and Colab keep for every cell
    code = getattr(fn, "__code__", None)
    if code is not None and Ellipsis in code.co_consts:
        return True
    import inspect
    try:
        lines = inspect.getsource(fn).splitlines()
    except Exception:
        return False
    for line in lines:
        if "..." in line.split("#")[0]:
            return True
    return False


def _part1_check(name, fn, needs):
    """Print one message about fn: works, not finished yet, or the first test it fails and a hint."""
    failure = _part1_first_failure(name, fn)
    if failure is None:
        print("✅ %s works on %d tests." % (name, len(_PART1_TESTS[name])))
        return
    not_finished = ("⏳ %s is not finished yet: replace each ... (Python calls it Ellipsis) with your code, "
                    "run its cell, then run this check again." % name)
    if _part1_has_gap(fn):
        print(not_finished)
        return
    for other, number in needs:
        other_fn = globals().get(other)
        if other_fn is None or _part1_first_failure(other, other_fn) is not None:
            print("⏳ %s relies on your %s from task %d, which does not work yet. "
                  "Get the check of task %d to ✅ first, then run this check again." % (name, other, number, number))
            return
    if failure["answer"] is Ellipsis or "ellipsis" in str(failure["error"]).lower():
        print(not_finished)
        return
    if failure["error"] is not None:
        print("❌ %s stopped with an error: %s" % (failure["call"], failure["error"]))
    elif failure["answer"] is None:
        print("❌ %s gave no answer (None). Does the function end with a return line?" % failure["call"])
    else:
        print("❌ %s gave %r, expected %r. %s." % (failure["call"], failure["answer"], failure["expected"], failure["why"]))
    print("   💡 Hint: " + _PART1_HINTS[name])


def check_inner_product(fn):
    """Task 1: tests inner_product on six pairs of customers."""
    try:
        _part1_check("inner_product", fn, [])
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell at the top again, then this cell." % error)


def check_nearest_customer(fn):
    """Task 2: tests nearest_customer on five small stores, with a tie and with scores below zero."""
    try:
        _part1_check("nearest_customer", fn, [("inner_product", 1)])
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell at the top again, then this cell." % error)


def check_predict(fn):
    """Task 3: tests predict on four stores."""
    try:
        _part1_check("predict", fn, [("inner_product", 1), ("nearest_customer", 2)])
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell at the top again, then this cell." % error)


print("✅ Setup complete.")

## Vera's store

Round 2 of the game, as plain Python lists. Each mark is a number:

| two ticks ✓✓ | a tick ✓ | neutral \~ | a cross ✗ | two crosses ✗✗ |
|:-:|:-:|:-:|:-:|:-:|
| 2, loved it | 1, liked it | 0 | -1, disliked it | -2, hated it |

`ratings` has one row per customer and one number per film. `vera` holds Vera's ratings of the same three
films, and `love_actually` every customer's rating of *Love Actually*, which Vera has not seen yet. Python
counts from 0: `ratings[3]` is Kai's row, `[1, 2, -2]`, and `vera[0]` is Vera's rating of *Jaws*, `2`.

In [ ]:
# Vera's store, from round 2 of the game
customers = ["Ivo", "Pia", "Tom", "Kai", "Jana", "Rosa"]
films = ["Jaws", "The Shining", "Notting Hill"]

# One row per customer, one rating per film, in the order of films
ratings = [
    [ 2, -2,  0],   # Ivo
    [ 2,  1,  2],   # Pia
    [-1, -2, -1],   # Tom
    [ 1,  2, -2],   # Kai
    [ 0,  1,  0],   # Jana
    [-2, -1,  2],   # Rosa
]

# Vera, the newcomer, has rated the same three films
vera = [2, 1, -1]

# Every customer's rating of Love Actually, in the order of customers. Vera has not seen it yet.
love_actually = [1, 2, -1, -2, 0, 2]

In [ ]:
#@title ▶ The store as a table { display-mode: "form" }
_part1_table(customers, films, ratings, "Vera", vera, "Love Actually", love_actually)

## 🎯 Task 1. The inner product

In the game you measured how much two customers agree: multiply their ratings film by film, then add up the
products. That number is their **inner product**. For Vera, `[2, 1, -1]`, and Kai, `[1, 2, -2]`:

`2*1 + 1*2 + (-1)*(-2) = 2 + 2 + 2 = 6`

Write `inner_product(customer_a, customer_b)`. It gets two lists of ratings of the same films and returns
their inner product. In Python, `*` multiplies.

### 🎯 Your turn

In [ ]:
# ✅ SOLUTION, task 1: the inner product of two customers
def inner_product(customer_a, customer_b):
    # customer_a and customer_b are two lists of ratings of the same films, in the same order.
    # Multiply the two ratings of each film, then add up all the products.
    total = 0
    for i in range(len(customer_a)):
        # customer_a[i] and customer_b[i] are the two customers' ratings of film number i
        product = customer_a[i] * customer_b[i]
        total = total + product
    return total

In [ ]:
check_inner_product(inner_product)

In [ ]:
# Vera's inner product with every customer, with your inner_product
for i in range(len(customers)):
    print(customers[i], inner_product(ratings[i], vera))

**Reading the result.** Kai has the largest inner product with Vera, 6: on every film Kai is on Vera's
side, only more or less strongly. Pia comes second with 3, although Pia gives exactly Vera's marks on two
films: Pia loves *Notting Hill*, which Vera dislikes. Rosa disagrees with Vera on every film and has the
smallest, -7. The customer with the largest inner product is Vera's **nearest neighbour**.

## 🎯 Task 2. The nearest customer

Write `nearest_customer(ratings, new_customer)`. It gets the store's table `ratings`, one row per customer,
and the ratings of a newcomer. It returns the **index** of the customer with the largest inner product with
the newcomer: that customer's row number, counted from 0. For Vera it should give 3, Kai's row.

If two customers tie, the first of them wins: a customer further down takes the lead only with a strictly
larger inner product.

### 🎯 Your turn

In [ ]:
# ✅ SOLUTION, task 2: the nearest customer
def nearest_customer(ratings, new_customer):
    # ratings has one row per customer. Return the index (the row number) of the customer
    # with the largest inner product with new_customer. On a tie, the first of them wins.

    # Start with the first customer, row 0, as the best so far
    best_index = 0
    best_score = inner_product(ratings[0], new_customer)

    # Then look at every customer in turn
    for i in range(len(ratings)):
        score = inner_product(ratings[i], new_customer)
        if score > best_score:
            # Customer i is the best so far: remember their score and their row
            best_score = score
            best_index = i

    return best_index

In [ ]:
check_nearest_customer(nearest_customer)

In [ ]:
nearest = nearest_customer(ratings, vera)
print("Vera's nearest neighbour is", customers[nearest], "in row", nearest)

## 🎯 Task 3. Predict a rating

Now the store's recipe from the game: find the newcomer's nearest neighbour, and copy that customer's
rating of a film the newcomer has not seen. Write `predict(ratings, new_film, new_customer)`. It gets:

- `ratings`: the store's table, one row per customer, as in task 2;
- `new_film`: every customer's rating of the film the newcomer has not seen, in the order of the rows, so
  `new_film[i]` is customer i's rating;
- `new_customer`: the newcomer's ratings of the films in `ratings`.

It returns the nearest customer's rating of the new film. For Vera the new film is *Love Actually*:
`predict(ratings, love_actually, vera)`.

### 🎯 Your turn

In [ ]:
# ✅ SOLUTION, task 3: predict the newcomer's rating of the new film
def predict(ratings, new_film, new_customer):
    # new_film[i] is customer i's rating of a film that new_customer has not seen.
    # Find the nearest customer, then return their rating of that film.
    nearest = nearest_customer(ratings, new_customer)
    return new_film[nearest]

In [ ]:
check_predict(predict)

In [ ]:
print("Prediction for Vera and Love Actually:", predict(ratings, love_actually, vera))

**Reading the result.** -2 is two crosses: Kai hated *Love Actually*, so the store predicts that Vera will
hate it too. And when Vera did watch it, she gave it two crosses. Pia, the customer with the most exact
matches, loved it: counting matches would have got Vera wrong.

### Back to round 1

The same functions work for any store. In round 1 of the game the cards had only ticks and crosses, so a
tick is 1 and a cross is -1. Run the cell to find Zoe's nearest neighbour and the prediction for
*Love Actually*.

In [ ]:
# Round 1 of the game, with the same three films: a tick is 1, a cross is -1
round1_customers = ["Lea", "Max", "Nina", "Omar"]
round1_ratings = [
    [ 1,  1, -1],   # Lea
    [-1, -1,  1],   # Max
    [ 1,  1, -1],   # Nina
    [-1, -1,  1],   # Omar
]
zoe = [-1, -1, 1]
round1_love_actually = [-1, 1, -1, 1]

nearest = nearest_customer(round1_ratings, zoe)
print("Zoe's nearest neighbour is", round1_customers[nearest], "in row", nearest)
print("Prediction for Zoe and Love Actually:", predict(round1_ratings, round1_love_actually, zoe))

**Reading the result.** Max and Omar rate exactly like Zoe, so both have inner product 3 with Zoe, and on a
tie the first one wins: Max. Max liked *Love Actually*, so the store predicts a tick, 1, and Zoe did like
it. With ticks and crosses only, every film where two customers agree adds 1 and every film where they
disagree adds -1: the inner product is the matches minus the mismatches. So in act 1, counting matches and
the inner product pick the same nearest neighbours.

## 🙋 Your turn to be the newcomer

Together, the three functions make a small recommender: `inner_product` measures how much two customers
agree, `nearest_customer` finds the customer who agrees most with a newcomer, and `predict` copies that
customer's rating of a film the newcomer has not seen.

Try it on yourself: put your own ratings of *Jaws*, *The Shining* and *Notting Hill* into `me`, from -2
(hated it) to 2 (loved it), with 0 for neutral or not seen, and run the cell.

In [ ]:
me = [1, -1, 2]   # your ratings of Jaws, The Shining and Notting Hill

nearest = nearest_customer(ratings, me)
print("Your nearest neighbour is", customers[nearest])
print("The store predicts your rating of Love Actually:", predict(ratings, love_actually, me))

## 🏆 Challenge: let the k nearest customers vote

Your `predict` lets a single customer decide: Kai's two crosses became Vera's prediction. Kai was right
about Vera, but a single customer can be a fluke. The next newcomer's nearest neighbour may have seen the
film on a bad day.

A careful store asks several customers. It finds the **k nearest customers**, the k customers with the
largest inner products with the newcomer, and lets them **vote**. Each of them votes with their own rating
of the new film, and the prediction is the rating that most of them gave. If two ratings get the same number
of votes, the rating of the nearer customer wins. With k = 1 only the nearest customer votes, and that is
your `predict` from task 3.

This part is a challenge: its cells give you less help than the tasks above. Each of its three tasks has
folded 💡 hints for when you are stuck: double-click a hint to open it.

In [ ]:
#@title Setup of the challenge (run this first) { display-mode: "form" }
# The plumbing of the challenge: its own copy of the data, the table of the store a month later, the checks
# of challenges 1 to 3, and the cell that tries every k. The checks keep working after you change the lists
# in the notebook. You do not need to read it. A check prints one short message and never stops the notebook.
try:
    from IPython.display import display as _k_display
except ImportError:
    _k_display = None

_K_MARKS = {2: '✓✓', 1: '✓', 0: '~', -1: '✗', -2: '✗✗'}
_K_COLOURS = {2: "#3B82D6", 1: "#3B82D6", 0: "#8C8C8C", -1: "#E0742A", -2: "#E0742A"}
_K_LEGEND = "✓✓ 2 loved it, ✓ 1 liked it, ~ 0 neutral, ✗ -1 disliked it, ✗✗ -2 hated it, ? not seen yet"
_K_FILMS = ['Jaws', 'The Shining', 'Notting Hill']

# Vera's store (round 2 of the game), round 1 (Zoe's), and the store a month later (invented)
_K_RATINGS = [[2, -2, 0], [2, 1, 2], [-1, -2, -1], [1, 2, -2], [0, 1, 0], [-2, -1, 2]]
_K_VERA = [2, 1, -1]
_K_LOVE_ACTUALLY = [1, 2, -1, -2, 0, 2]
_K_ROUND1_RATINGS = [[1, 1, -1], [-1, -1, 1], [1, 1, -1], [-1, -1, 1]]
_K_ZOE = [-1, -1, 1]
_K_ROUND1_LOVE_ACTUALLY = [-1, 1, -1, 1]
_K_RATINGS_LATER = [[2, -2, 0], [2, 1, 2], [-1, -2, -1], [1, 2, -2], [0, 1, 0], [-2, -1, 2], [2, 1, -1], [2, 2, -2], [1, 1, -1], [-1, 0, 2]]
_K_NOAH = [2, 2, -2]
_K_LOVE_ACTUALLY_LATER = [1, 2, -1, -2, 0, 2, -2, 2, -2, 2]

# Each test: the inputs, the expected answer, the call as you would type it, and why that is the answer.
_K_TESTS = {
    "inner_product": [
        (([2, 1, -1], [2, -2, 0]), 2,
         'inner_product(vera, ratings[0])',
         ''),
        (([2, 1, -1], [2, 1, 2]), 3,
         'inner_product(vera, ratings[1])',
         ''),
        (([2, 1, -1], [-1, -2, -1]), -3,
         'inner_product(vera, ratings[2])',
         ''),
        (([2, 1, -1], [1, 2, -2]), 6,
         'inner_product(vera, ratings[3])',
         ''),
        (([2, 1, -1], [0, 1, 0]), 1,
         'inner_product(vera, ratings[4])',
         ''),
        (([2, 1, -1], [-2, -1, 2]), -7,
         'inner_product(vera, ratings[5])',
         ''),
    ],
    "k_nearest_customers": [
        (([[2, -2, 0], [2, 1, 2], [-1, -2, -1], [1, 2, -2], [0, 1, 0], [-2, -1, 2]], [2, 1, -1], 1), [3],
         'k_nearest_customers(ratings, vera, 1)',
         "With k = 1 the list holds one index, the nearest customer's: Kai, in row 3"),
        (([[2, -2, 0], [2, 1, 2], [-1, -2, -1], [1, 2, -2], [0, 1, 0], [-2, -1, 2]], [2, 1, -1], 3), [3, 1, 0],
         'k_nearest_customers(ratings, vera, 3)',
         "Vera's inner products are Kai 6, Pia 3, Ivo 2, Jana 1, Tom -3 and Rosa -7: the three largest, the nearest first"),
        (([[2, -2, 0], [2, 1, 2], [-1, -2, -1], [1, 2, -2], [0, 1, 0], [-2, -1, 2]], [2, 1, -1], 6), [3, 1, 0, 4, 2, 5],
         'k_nearest_customers(ratings, vera, 6)',
         'With k = 6 every customer is in the list, the nearest first: Kai 6, Pia 3, Ivo 2, Jana 1, Tom -3 and Rosa -7'),
        (([[2, -2, 0], [2, 1, 2], [-1, -2, -1], [1, 2, -2], [0, 1, 0], [-2, -1, 2], [2, 1, -1], [2, 2, -2], [1, 1, -1], [-1, 0, 2]], [2, 2, -2], 3), [7, 3, 6],
         'k_nearest_customers(ratings_later, noah, 3)',
         "Noah's three nearest customers are Ugo, Kai and Vera, with inner products 12, 10 and 8"),
        (([[2, 2], [1, 1], [-1, -1]], [1, 1], 2), [0, 1],
         'k_nearest_customers([[2, 2], [1, 1], [-1, -1]], [1, 1], 2)',
         'Here the first customer, row 0, is the nearest, and each customer appears only once in the list'),
        (([[1, 0], [0, 1], [1, 0]], [1, 0], 2), [0, 2],
         'k_nearest_customers([[1, 0], [0, 1], [1, 0]], [1, 0], 2)',
         'Rows 0 and 2 tie with 1: the first of them wins the first round, the other one the second'),
        (([[-2, -2], [-1, 0], [0, -2]], [1, 1], 2), [1, 2],
         'k_nearest_customers([[-2, -2], [-1, 0], [0, -2]], [1, 1], 2)',
         'All the inner products are below zero here (-4, -1 and -2), and the largest still wins: -1, then -2'),
    ],
    "majority_vote": [
        (([2],), 2,
         'majority_vote([2])',
         'With one vote, that vote is the prediction'),
        (([2, -2, -2],), -2,
         'majority_vote([2, -2, -2])',
         '-2 appears twice and 2 once'),
        (([-2, 2, 1],), -2,
         'majority_vote([-2, 2, 1])',
         'Each vote appears once, a tie, so the vote that comes first wins'),
        (([1, 0, 0, 1],), 1,
         'majority_vote([1, 0, 0, 1])',
         '1 and 0 appear twice each, a tie, and 1 comes first'),
        (([0, -1, 2, -1, 2],), -1,
         'majority_vote([0, -1, 2, -1, 2])',
         '-1 and 2 appear twice each, and -1 comes first'),
        (([2, 1, 1, -2, -2, -2],), -2,
         'majority_vote([2, 1, 1, -2, -2, -2])',
         '-2 appears three times, more often than any other vote, although it comes last'),
    ],
    "predict_k": [
        (([[2, -2, 0], [2, 1, 2], [-1, -2, -1], [1, 2, -2], [0, 1, 0], [-2, -1, 2]], [1, 2, -1, -2, 0, 2], [2, 1, -1], 1), -2,
         'predict_k(ratings, love_actually, vera, 1)',
         'With k = 1 only Kai votes, so the prediction is his rating, -2, as with predict in task 3'),
        (([[2, -2, 0], [2, 1, 2], [-1, -2, -1], [1, 2, -2], [0, 1, 0], [-2, -1, 2]], [1, 2, -1, -2, 0, 2], [2, 1, -1], 3), -2,
         'predict_k(ratings, love_actually, vera, 3)',
         'Kai, Pia and Ivo vote -2, 2 and 1: three different votes, a tie, so the nearest, Kai, wins'),
        (([[2, -2, 0], [2, 1, 2], [-1, -2, -1], [1, 2, -2], [0, 1, 0], [-2, -1, 2]], [1, 2, -1, -2, 0, 2], [2, 1, -1], 6), 2,
         'predict_k(ratings, love_actually, vera, 6)',
         'With k = 6 all six customers vote, -2, 2, 1, 0, -1 and 2, and only 2 appears twice'),
        (([[1, 1, -1], [-1, -1, 1], [1, 1, -1], [-1, -1, 1]], [-1, 1, -1, 1], [-1, -1, 1], 3), 1,
         'predict_k([[1, 1, -1], [-1, -1, 1], [1, 1, -1], [-1, -1, 1]], [-1, 1, -1, 1], [-1, -1, 1], 3)',
         "Round 1: Zoe's three nearest are Max, Omar and Lea, who vote 1, 1 and -1"),
        (([[2, -2, 0], [2, 1, 2], [-1, -2, -1], [1, 2, -2], [0, 1, 0], [-2, -1, 2], [2, 1, -1], [2, 2, -2], [1, 1, -1], [-1, 0, 2]], [1, 2, -1, -2, 0, 2, -2, 2, -2, 2], [2, 2, -2], 1), 2,
         'predict_k(ratings_later, love_actually_later, noah, 1)',
         'With k = 1 only Ugo votes, and he gave Love Actually 2'),
        (([[2, -2, 0], [2, 1, 2], [-1, -2, -1], [1, 2, -2], [0, 1, 0], [-2, -1, 2], [2, 1, -1], [2, 2, -2], [1, 1, -1], [-1, 0, 2]], [1, 2, -1, -2, 0, 2, -2, 2, -2, 2], [2, 2, -2], 3), -2,
         'predict_k(ratings_later, love_actually_later, noah, 3)',
         'Ugo, Kai and Vera vote 2, -2 and -2: -2 wins, two votes against one'),
        (([[2, -2, 0], [2, 1, 2], [-1, -2, -1], [1, 2, -2], [0, 1, 0], [-2, -1, 2], [2, 1, -1], [2, 2, -2], [1, 1, -1], [-1, 0, 2]], [1, 2, -1, -2, 0, 2, -2, 2, -2, 2], [2, 2, -2], 10), 2,
         'predict_k(ratings_later, love_actually_later, noah, 10)',
         'With k = 10 everybody votes: 2 four times, -2 three times'),
    ],
}

_K_HINTS = {
    "k_nearest_customers": "in each of the k rounds, skip the customers already in chosen (if i not in chosen:), "
                           "keep the one with the largest inner product, and append its index to chosen.",
    "majority_vote": "for each vote, count how often it appears in votes, and keep the vote with the strictly "
                     "largest count, so that on a tie the earlier vote stays.",
    "predict_k": "nearest = k_nearest_customers(ratings, new_customer, k); the votes are new_film[i] for each i "
                 "in nearest; return majority_vote(votes).",
}
_K_SUCCESS = {'k_nearest_customers': "Vera's three nearest customers are Kai, Pia and Ivo; Noah's are Ugo, Kai and Vera.", 'majority_vote': 'On a tie, the vote that comes first wins.', 'predict_k': 'For Vera and k = 1 it gives -2, as predict did in task 3. For Noah, k = 1 gives 2 and k = 3 gives -2.'}


def _k_mark(rating, as_html):
    """One cell of a table: the mark and the number, or ? when the rating is not known."""
    if rating is None:
        return "?"
    mark = _K_MARKS.get(rating, "")
    if as_html:
        colour = _K_COLOURS.get(rating, "inherit")
        return '<b style="color:%s">%s</b>&nbsp;%s' % (colour, mark, rating)
    return "%-2s %2s" % (mark, rating)


def _k_table(customers, films, ratings, newcomer, newcomer_ratings, new_film, new_film_ratings):
    """Show a store as a table of marks and numbers, with the newcomer in the last row."""
    from html import escape
    try:
        names = list(customers) + [newcomer]
        heads = list(films) + [new_film]
        rows = []
        for i in range(len(customers)):
            rows.append(list(ratings[i]) + [new_film_ratings[i]])
        rows.append(list(newcomer_ratings) + [None])

        # Plain text, for screens that cannot show the table
        name_width = max(len(name) for name in names) + 3
        widths = [max(len(head), 6) + 3 for head in heads]
        lines = [" " * name_width + "".join(heads[j].ljust(widths[j]) for j in range(len(heads)))]
        for r in range(len(rows)):
            cells = "".join(_k_mark(rows[r][j], False).ljust(widths[j]) for j in range(len(heads)))
            lines.append(names[r].ljust(name_width) + cells)
        text = "\n".join(line.rstrip() for line in lines) + "\n\n" + _K_LEGEND

        # The same table in HTML: the new film's column and the newcomer's row are set apart by a stronger line
        thin = "border-bottom:1px solid rgba(128,128,128,0.35);"
        strong = "2px solid rgba(128,128,128,0.7);"
        html = '<table style="border-collapse:collapse;font-size:15px;margin:4px 0 6px 0">'
        html += '<tr><th style="padding:5px 12px;%s"></th>' % thin
        for j in range(len(heads)):
            left = "border-left:" + strong if j == len(heads) - 1 else ""
            html += '<th style="padding:5px 12px;text-align:center;%s%s">%s</th>' % (thin, left, escape(str(heads[j])))
        html += "</tr>"
        for r in range(len(rows)):
            is_newcomer = r == len(rows) - 1
            top = "border-top:" + strong if is_newcomer else ""
            name = escape(str(names[r]))
            if is_newcomer:
                name = "<b>%s</b>" % name
            html += '<tr><td style="padding:5px 12px;text-align:left;%s%s">%s</td>' % (thin, top, name)
            for j in range(len(heads)):
                left = "border-left:" + strong if j == len(heads) - 1 else ""
                html += '<td style="padding:5px 12px;text-align:center;%s%s%s">%s</td>' % (
                    thin, top, left, _k_mark(rows[r][j], True))
            html += "</tr>"
        html += "</table>"
        html += '<div style="font-size:13px;opacity:0.8">%s</div>' % escape(_K_LEGEND)

        if _k_display is None:
            print(text)
        else:
            _k_display({"text/plain": text, "text/html": html}, raw=True)
    except Exception as error:
        print("The table could not be drawn (%s). Run the cell with the data again, then this one." % error)


def _k_call(function, inputs):
    """Run a participant's function on fresh copies of the inputs, with its prints silenced.
    Gives back (answer, error), where error is None when the call went through. Never raises."""
    import contextlib, copy, io
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            answer = function(*copy.deepcopy(inputs))
        return answer, None
    except Exception as error:
        return None, "%s: %s" % (type(error).__name__, error)


def _k_first_failure(name, function):
    """The first test of name that function fails, or None when it passes them all."""
    for inputs, expected, call, why in _K_TESTS[name]:
        answer, error = _k_call(function, inputs)
        try:
            right = bool(error is None and answer == expected)
        except Exception:
            right = False
        if not right:
            return {"inputs": inputs, "answer": answer, "error": error, "expected": expected, "call": call,
                    "why": why}
    return None


def _k_has_gap(function):
    """True when the code of function still holds a ... to fill in (comments do not count)."""
    import inspect
    try:
        lines = inspect.getsource(function).splitlines()
    except Exception:
        return False
    for line in lines:
        if "..." in line.split("#")[0]:
            return True
    return False


def _k_hint(name, failure):
    """The hint for a wrong answer: a precise one when the answer shows the slip, else the general one."""
    answer, inputs, expected = failure["answer"], failure["inputs"], failure["expected"]
    if failure["error"] is not None or answer is None:
        return _K_HINTS[name]
    try:
        if name == "k_nearest_customers":
            k = inputs[2]
            if not isinstance(answer, list):
                return "return the list chosen, with the row numbers (indices) of the customers."
            if len(answer) == 0:
                return ("your list is empty: run k rounds, and in each round append the index of the customer "
                        "you choose to chosen.")
            for item in answer:
                if not isinstance(item, int):
                    return "chosen should hold row numbers (indices), whole numbers such as 3 for Kai."
            for i in range(len(answer)):
                if answer[i] in answer[:i]:
                    return ("customer %d appears twice: skip the customers already in chosen (if i not in chosen:), "
                            "also for the customer each round starts with." % answer[i])
            if len(answer) != k:
                return "your list holds %d indices, and with k = %d it should hold %d." % (len(answer), k, k)
            if sorted(answer) == sorted(expected):
                return ("these are the right customers in another order. The nearest comes first, and on a tie the "
                        "earlier customer stays: replace the best only when a score is strictly larger (>).")
        if name == "majority_vote" and answer not in inputs[0]:
            return "return one of the votes, the rating that appears most often: its count only helps you choose."
        if name == "predict_k" and isinstance(answer, list):
            return "return a single rating, the majority vote of the list of votes: majority_vote(votes)."
    except Exception:
        pass
    return _K_HINTS[name]


def _k_check(name, function, needs):
    """Print one message about function: it passes, it still has a gap, or the first test it fails and a hint."""
    failure = _k_first_failure(name, function)
    if failure is None:
        print("✅ %s works on %d tests. %s" % (name, len(_K_TESTS[name]), _K_SUCCESS[name]))
        return
    not_finished = ("⏳ %s is not finished yet: replace each ... (Python calls it Ellipsis) with your code, "
                    "run its cell, then run this check again." % name)
    if _k_has_gap(function):
        print(not_finished)
        return
    for other, place in needs:
        other_function = globals().get(other)
        if other_function is None or _k_first_failure(other, other_function) is not None:
            print("⏳ %s relies on your %s from %s, which does not work yet. "
                  "Get the check of %s to ✅ first, then run this check again." % (name, other, place, place))
            return
    if failure["answer"] is Ellipsis or "ellipsis" in str(failure["error"]).lower():
        print(not_finished)
        return
    if failure["error"] is not None:
        print("❌ %s stopped with an error: %s" % (failure["call"], failure["error"]))
    elif failure["answer"] is None:
        print("❌ %s gave no answer (None). Does the function end with a return line?" % failure["call"])
    else:
        print("❌ %s gave %r, expected %r. %s." % (failure["call"], failure["answer"], failure["expected"],
                                                   failure["why"]))
    print("   💡 Hint: " + _k_hint(name, failure))


def check_k_nearest_customers(fn):
    """Challenge 1: tests k_nearest_customers on Vera's store, the store a month later and three small stores."""
    try:
        _k_check("k_nearest_customers", fn, [("inner_product", "task 1")])
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of the challenge again, then this cell." % error)


def check_majority_vote(fn):
    """Challenge 2: tests majority_vote on six lists of votes, with ties."""
    try:
        _k_check("majority_vote", fn, [])
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of the challenge again, then this cell." % error)


def check_predict_k(fn):
    """Challenge 3: tests predict_k on Vera's store, round 1 and the store a month later."""
    try:
        _k_check("predict_k", fn, [("inner_product", "task 1"), ("k_nearest_customers", "challenge 1"),
                                   ("majority_vote", "challenge 2")])
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of the challenge again, then this cell." % error)


def _k_every_k(customers, ratings, newcomer, newcomer_ratings, new_film, new_film_ratings):
    """For every k, the prediction of your predict_k and the votes of your k nearest customers."""
    import contextlib, copy, io
    lines = []
    try:
        nearest_function = globals()["k_nearest_customers"]
        predict_function = globals()["predict_k"]
        for k in range(1, len(ratings) + 1):
            with contextlib.redirect_stdout(io.StringIO()):
                nearest = nearest_function(copy.deepcopy(ratings), copy.deepcopy(newcomer_ratings), k)
                prediction = predict_function(copy.deepcopy(ratings), copy.deepcopy(new_film_ratings),
                                              copy.deepcopy(newcomer_ratings), k)
            if len(nearest) != k or prediction not in _K_MARKS:
                raise ValueError("not ready yet")
            votes = ", ".join("%s %s" % (customers[i], _K_MARKS[new_film_ratings[i]]) for i in nearest)
            lines.append("k = %2d   prediction %s   votes: %s" % (k, _k_mark(prediction, False), votes))
    except Exception:
        print("⏳ This cell uses your k_nearest_customers and predict_k. "
              "Run it again once the checks of challenges 1 and 3 show ✅.")
        return
    print("The store's prediction for %s and %s, for every k:" % (newcomer, new_film))
    print()
    print("\n".join(lines))
    print()
    print(_K_LEGEND)


print("✅ Setup of the challenge complete.")

### The store a month later

The store has grown. Vera is a regular now, and her two crosses for *Love Actually* are in the table. Three
new regulars have joined, Ugo, Sam and Mia, and a newcomer, **Noah**, has rated the three films. Like
everything in this store, the new rows are invented.

In [ ]:
# The store a month later, with the same three films as before
customers_later = ["Ivo", "Pia", "Tom", "Kai", "Jana", "Rosa", "Vera", "Ugo", "Sam", "Mia"]

# One row per customer, one rating per film, in the order of films
ratings_later = [
    [ 2, -2,  0],   # Ivo
    [ 2,  1,  2],   # Pia
    [-1, -2, -1],   # Tom
    [ 1,  2, -2],   # Kai
    [ 0,  1,  0],   # Jana
    [-2, -1,  2],   # Rosa
    [ 2,  1, -1],   # Vera, a regular now
    [ 2,  2, -2],   # Ugo, new
    [ 1,  1, -1],   # Sam, new
    [-1,  0,  2],   # Mia, new
]

# Noah, the newcomer, has rated the same three films
noah = [2, 2, -2]

# Every customer's rating of Love Actually, in the order of customers_later. Noah has not seen it yet.
love_actually_later = [1, 2, -1, -2, 0, 2, -2, 2, -2, 2]

In [ ]:
#@title ▶ The store a month later, as a table { display-mode: "form" }
_k_table(customers_later, _K_FILMS, ratings_later, "Noah", noah, "Love Actually", love_actually_later)

💭 **Think first.** Who is Noah's nearest neighbour, and what did that customer think of *Love Actually*?
What did the next two nearest think?

## 🎯 Challenge 1. The k nearest customers

Write `k_nearest_customers(ratings, new_customer, k)`. It returns a list with the indices of the k customers
with the largest inner products with `new_customer`, the nearest first. On a tie, the first of them wins, as
in task 2.

For example, Vera's inner products are Kai 6, Pia 3, Ivo 2, Jana 1, Tom -3 and Rosa -7, so `k_nearest_customers(ratings, vera, 3)` gives `[3, 1, 0]`:
Kai, Pia and Ivo.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for challenge 1 (double-click to open) { display-mode: "form" }
# Run the search of task 2 k times: one round for each customer you choose.
# Each round looks only at the customers who are not in chosen yet (if i not in chosen:),
# keeps the one with the largest inner product with new_customer,
# and appends that customer's index to chosen.
# The first round finds the nearest customer, the second round the next nearest, and so on.

In [ ]:
#@title 💡 Hint 2 for challenge 1: the shape of the code (double-click to open) { display-mode: "form" }
# These lines go between chosen = [] and return chosen.
#
# for step in range(k):               # k rounds, one for each customer you choose
#     best_index = -1                 # -1 means: nobody found yet in this round
#     best_score = 0
#     for i in range(len(ratings)):
#         if i not in chosen:
#             score = ...             # the inner product of customer i and new_customer
#             if best_index == -1:
#                 ...                 # the first candidate of the round: keep i and its score
#             elif score > best_score:
#                 ...                 # a more similar candidate: keep i and its score instead
#     chosen.append(best_index)

In [ ]:
# ✅ SOLUTION, challenge 1: the k nearest customers
def k_nearest_customers(ratings, new_customer, k):
    # Return a list with the indices (row numbers) of the k customers with the largest
    # inner products with new_customer, the nearest first. On a tie, the first of them wins.
    chosen = []
    # One round for each customer we choose: k rounds
    for step in range(k):
        # The search of task 2, among the customers who are not chosen yet
        best_index = -1      # -1 means: nobody found yet in this round
        best_score = 0       # the first candidate of the round replaces it
        for i in range(len(ratings)):
            if i not in chosen:
                score = inner_product(ratings[i], new_customer)
                if best_index == -1:
                    # The first candidate of this round
                    best_index = i
                    best_score = score
                elif score > best_score:
                    # A more similar candidate. On a tie, the earlier customer stays.
                    best_index = i
                    best_score = score
        # The winner of this round joins the list
        chosen.append(best_index)
    return chosen

In [ ]:
check_k_nearest_customers(k_nearest_customers)

## 🎯 Challenge 2. The majority vote

Write `majority_vote(votes)`. `votes` is a list of ratings, the nearest customer's vote first. Return the
rating that appears most often. On a tie, the rating that comes first in the list wins: it belongs to the
nearer customer.

For example, `majority_vote([2, -2, -2])` gives -2, two votes against one. `majority_vote([-2, 2, 1])` gives
-2 as well: three different votes make a tie, and the first one wins.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for challenge 2 (double-click to open) { display-mode: "form" }
# For each vote in votes, count how many times it appears in votes: a loop inside the loop.
# Remember the most common vote so far, and its count, in two variables:
# best_vote and best_count.
# Replace them only when a count is strictly bigger (>). An equal count keeps the
# earlier vote, and that is the tie rule.

In [ ]:
#@title 💡 Hint 2 for challenge 2: the shape of the code (double-click to open) { display-mode: "form" }
# best_vote = votes[0]
# best_count = 0
# for vote in votes:
#     count = 0
#     for other_vote in votes:
#         ...                         # add 1 to count when other_vote equals vote
#     if count > best_count:
#         ...                         # keep vote and count as the best so far
# return best_vote

In [ ]:
# ✅ SOLUTION, challenge 2: the majority vote
def majority_vote(votes):
    # votes is a list of ratings, the nearest customer's first. Return the rating that
    # appears most often. On a tie, the one that comes first in the list wins.
    best_vote = votes[0]
    best_count = 0
    for vote in votes:
        # How many customers gave this same rating?
        count = 0
        for other_vote in votes:
            if other_vote == vote:
                count = count + 1
        # Keep it if it is more common than the best so far.
        # An equal count keeps the earlier vote: the nearer customer wins a tie.
        if count > best_count:
            best_vote = vote
            best_count = count
    return best_vote

In [ ]:
check_majority_vote(majority_vote)

## 🎯 Challenge 3. Predict with the k nearest customers

Write `predict_k(ratings, new_film, new_customer, k)`, your `predict` from task 3 with a number k. Find the k
nearest customers, put their ratings of the new film in a list of votes (customer i's rating is
`new_film[i]`), and return the majority vote. Your two functions above do most of the work.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint for challenge 3 (double-click to open) { display-mode: "form" }
# 1. nearest = k_nearest_customers(ratings, new_customer, k)
# 2. Start with votes = [] and, for each index i in nearest, append new_film[i] to votes.
# 3. Return majority_vote(votes).

In [ ]:
# ✅ SOLUTION, challenge 3: predict with the k nearest customers
def predict_k(ratings, new_film, new_customer, k):
    # new_film[i] is customer i's rating of a film that new_customer has not seen.
    # Find the k nearest customers, collect their ratings of that film, and return the majority vote.
    nearest = k_nearest_customers(ratings, new_customer, k)
    # Their ratings of the new film are the votes
    votes = []
    for i in nearest:
        votes.append(new_film[i])
    # The rating that most of them gave is the prediction
    return majority_vote(votes)

In [ ]:
check_predict_k(predict_k)

### Noah, one k at a time

💭 **Think first.** What will the store predict for Noah with k = 1? With k = 3? And with k = 10, when every
customer in the store votes? The next cell asks your `predict_k` for every k from 1 to 10.

In [ ]:
#@title ▶ Noah's prediction for every k, with your functions { display-mode: "form" }
_k_every_k(customers_later, ratings_later, "Noah", noah, "Love Actually", love_actually_later)

**Reading the result.** With k = 1 the store follows Ugo alone. He rated the three films exactly like Noah,
and he loved *Love Actually* (maybe he watches it every Christmas). With k = 2 the vote is one against one,
and Ugo, the nearer customer, still wins. From k = 3 to k = 8 the customers who hated it, Kai and Vera and
then Sam, have the most votes: two crosses. With k = 9 and k = 10 nearly the whole store votes, the fans of
*Love Actually* catch up, and the prediction flips back to two ticks. With k = 10 every newcomer would get
that same prediction, whatever their taste.

## What k does

A small k follows one person: if that person is a fluke, so is the prediction. A large k follows the crowd,
and the newcomer's taste stops mattering. In between, a handful of customers like the newcomer decide. A
store can choose k by trying several values on customers whose ratings it already knows, and keeping the k
that guesses right most often.

This is how "people like you also liked" recommenders worked for years: find the customers most like you,
and let them vote.